# Module 4 Homework (2026 Cohort) - patch cells


## Question 1 - Fewer features (drop `ticker` and `month_wom` from CATEGORICAL)

In [9]:
!pip install yfinance

In [11]:
# IMPORTS
import numpy as np
import pandas as pd

#Fin Data Sources
import yfinance as yf
# import pandas_datareader as pdr

#Data viz
import plotly.graph_objs as go
import plotly.graph_objects as go
import plotly.express as px

import time
from datetime import date

# for graphs
import matplotlib.pyplot as plt
import seaborn as sns


# Imports form ML (Decision Trees)
from sklearn.impute import SimpleImputer
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score

# Disable SettingWithCopyWarning
pd.options.mode.chained_assignment = None  # default='warn'

In [10]:

# read files shared via google-drive-link
# https://stackoverflow.com/questions/62759748/downloading-data-from-a-shared-google-drive-link-in-google-colab

!pip uninstall gdown -y && pip install gdown
!gdown -V


Found existing installation: gdown 5.2.2
Uninstalling gdown-5.2.2:
  Successfully uninstalled gdown-5.2.2
gdown 6.5.1 at /usr/local/lib/python3.13/dist-packages


In [4]:
# https://stackoverflow.com/questions/62759748/downloading-data-from-a-shared-google-drive-link-in-google-colab
# precaculated data from Module 2: https://drive.google.com/file/d/1m3Qisfs2XfWk6Sw_Uk5kHLWqwQ0q8SKb/view?usp=sharing
!gdown https://drive.google.com/file/d/1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5/view?usp=sharing -O /content/

/usr/local/lib/python3.13/dist-packages/gdown/parse_url.py:48: UserWarning: You specified a Google Drive link that is not the correct link to download a file. You might want to try `--fuzzy` option or the following url: https://drive.google.com/uc?id=1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5
  warnings.warn(
Downloading...
From: https://drive.google.com/file/d/1oQSUMCs2DyQQIh8Y62UhrT00cIsE9Sr5/view?usp=sharing
To: /content/view?usp=sharing
84.3kB [00:00, 50.2MB/s]


In [12]:
df_full = pd.read_parquet("/content/sample_data/stocks_df_combined_2026_09_18.parquet.brotli", )

In [13]:

df_full.info()

<class 'pandas.core.frame.DataFrame'>
Index: 240759 entries, 0 to 6018
Columns: 203 entries, Open to growth_btc_usd_365d
dtypes: datetime64[ns](3), float64(129), int32(64), int64(5), object(2)
memory usage: 315.9+ MB


In [14]:

# growth indicators (but not future growth)
GROWTH = [g for g in df_full.keys() if (g.find('growth_')==0)&(g.find('future')<0)]
GROWTH

['growth_1d',
 'growth_3d',
 'growth_7d',
 'growth_30d',
 'growth_90d',
 'growth_365d',
 'growth_dax_1d',
 'growth_dax_3d',
 'growth_dax_7d',
 'growth_dax_30d',
 'growth_dax_90d',
 'growth_dax_365d',
 'growth_snp500_1d',
 'growth_snp500_3d',
 'growth_snp500_7d',
 'growth_snp500_30d',
 'growth_snp500_90d',
 'growth_snp500_365d',
 'growth_dji_1d',
 'growth_dji_3d',
 'growth_dji_7d',
 'growth_dji_30d',
 'growth_dji_90d',
 'growth_dji_365d',
 'growth_epi_1d',
 'growth_epi_3d',
 'growth_epi_7d',
 'growth_epi_30d',
 'growth_epi_90d',
 'growth_epi_365d',
 'growth_gold_1d',
 'growth_gold_3d',
 'growth_gold_7d',
 'growth_gold_30d',
 'growth_gold_90d',
 'growth_gold_365d',
 'growth_wti_oil_1d',
 'growth_wti_oil_3d',
 'growth_wti_oil_7d',
 'growth_wti_oil_30d',
 'growth_wti_oil_90d',
 'growth_wti_oil_365d',
 'growth_brent_oil_1d',
 'growth_brent_oil_3d',
 'growth_brent_oil_7d',
 'growth_brent_oil_30d',
 'growth_brent_oil_90d',
 'growth_brent_oil_365d',
 'growth_btc_usd_1d',
 'growth_btc_usd_3d',


In [15]:
df_full.keys()

Index(['Open', 'High', 'Low', 'Close_x', 'Volume', 'Dividends', 'Stock Splits',
       'Ticker', 'Year', 'Month',
       ...
       'growth_brent_oil_7d', 'growth_brent_oil_30d', 'growth_brent_oil_90d',
       'growth_brent_oil_365d', 'growth_btc_usd_1d', 'growth_btc_usd_3d',
       'growth_btc_usd_7d', 'growth_btc_usd_30d', 'growth_btc_usd_90d',
       'growth_btc_usd_365d'],
      dtype='object', length=203)

In [16]:
OHLCV = ['Open','High','Low','Close_x','Volume']

In [17]:
CATEGORICAL = ['Month', 'Weekday', 'Ticker', 'ticker_type']

In [18]:

TO_PREDICT = [g for g in df_full.keys() if (g.find('future')>=0)]
TO_PREDICT

['growth_future_30d', 'is_positive_growth_30d_future']

In [19]:

# we define dummy variables on Dividends and Stock Splits events later, but drop the original abs. values
TO_DROP = ['Year','Date','index_x', 'index_y', 'index', 'Quarter','Close_y','Dividends','Stock Splits'] + CATEGORICAL + OHLCV
TO_DROP

['Year',
 'Date',
 'index_x',
 'index_y',
 'index',
 'Quarter',
 'Close_y',
 'Dividends',
 'Stock Splits',
 'Month',
 'Weekday',
 'Ticker',
 'ticker_type',
 'Open',
 'High',
 'Low',
 'Close_x',
 'Volume']

In [20]:
# let's define on more custom numerical features
# Add a small constant to avoid log(0)
df_full['ln_volume'] = df_full.Volume.apply(lambda x: np.log(x+ 1e-6))


# define columns on Dividends or Stock Splits
df_full['div_payout'] = (df_full.Dividends>0).astype(int)
df_full['stock_split'] = (df_full['Stock Splits']>0).astype(int)

In [21]:
CUSTOM_NUMERICAL = ['SMA10', 'SMA20', 'growing_moving_average', 'high_minus_low_relative','volatility', 'ln_volume','div_payout','stock_split']

In [22]:
TECHNICAL_INDICATORS = ['adx', 'adxr', 'apo', 'aroon_1','aroon_2', 'aroonosc',
 'bop', 'cci', 'cmo','dx', 'macd', 'macdsignal', 'macdhist', 'macd_ext',
 'macdsignal_ext', 'macdhist_ext', 'macd_fix', 'macdsignal_fix',
 'macdhist_fix', 'mfi', 'minus_di', 'mom', 'plus_di', 'dm', 'ppo',
 'roc', 'rocp', 'rocr', 'rocr100', 'rsi', 'slowk', 'slowd', 'fastk',
 'fastd', 'fastk_rsi', 'fastd_rsi', 'trix', 'ultosc', 'willr',
 'ad', 'adosc', 'obv', 'atr', 'natr', 'ht_dcperiod', 'ht_dcphase',
 'ht_phasor_inphase', 'ht_phasor_quadrature', 'ht_sine_sine', 'ht_sine_leadsine',
 'ht_trendmod', 'avgprice', 'medprice', 'typprice', 'wclprice']


TECHNICAL_PATTERNS = [g for g in df_full.keys() if g.find('cdl')>=0]
print(f'Technical patterns count = {len(TECHNICAL_PATTERNS)}, examples = {TECHNICAL_PATTERNS[0:5]}')

Technical patterns count = 61, examples = ['cdl2crows', 'cdl3blackrows', 'cdl3inside', 'cdl3linestrike', 'cdl3outside']


In [23]:
MACRO = ['gdppot_us_yoy', 'gdppot_us_qoq', 'cpi_core_yoy', 'cpi_core_mom', 'FEDFUNDS',
 'DGS1', 'DGS5', 'DGS10']


NUMERICAL = GROWTH + TECHNICAL_INDICATORS + TECHNICAL_PATTERNS + CUSTOM_NUMERICAL + MACRO


# CHECK: NO OTHER INDICATORS LEFT
OTHER = [k for k in df_full.keys() if k not in OHLCV + CATEGORICAL + NUMERICAL + TO_DROP + TO_PREDICT]
OTHER

df_full.Ticker.nunique()

33

In [24]:

# truncated df_full with 25 years of data (and defined growth variables)
df = df_full[df_full.Date>='2000-01-01']
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 202292 entries, 3490 to 6018
Columns: 206 entries, Open to stock_split
dtypes: datetime64[ns](3), float64(130), int32(64), int64(7), object(2)
memory usage: 270.1+ MB


In [25]:
# tickers, min-max date, count of daily observations
df.groupby(['Ticker'])['Date'].agg(['min','max','count'])

,min,max,count
Ticker,,,
AAPL,2000-01-03,2026-09-18,6718
ACN,2001-07-19,2026-09-18,6329
AMZN,2000-01-03,2026-09-18,6718
ASML,2000-01-03,2026-09-18,6718
AVGO,2009-08-06,2026-09-18,4306
BHARTIARTL.NS,2002-07-01,2026-09-18,6016
BRK-B,2000-01-03,2026-09-18,6718
CDI.PA,2000-01-03,2026-09-18,6863
GOOG,2004-08-19,2026-09-18,5556


In [27]:
# dummy variables are not generated from Date and numeric variables

# df.loc[:,'Month'] = df['Month'].dt.strftime('%B').astype('string')

df.loc[:,'Month']= pd.to_datetime(df['Month'], format='%m').dt.strftime('%B')

df.loc[:,'Weekday'] = df['Weekday'].astype('string')
# .astype(str)

/tmp/ipykernel_3641/2619569166.py:5: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['January' 'January' 'January' ... 'September' 'September' 'September']' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  df.loc[:,'Month']= pd.to_datetime(df['Month'], format='%m').dt.strftime('%B')
/tmp/ipykernel_3641/2619569166.py:7: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '<StringArray>
['0', '1', '2', '3', '4', '0', '1', '2', '3', '4',
 ...
 '0', '1', '2', '3', '4', '0', '1', '2', '3', '4']
Length: 202292, dtype: string' has dtype incompatible with int32, please explicitly cast to a compatible dtype first.
  df.loc[:,'Weekday'] = df['Weekday'].astype('string')


In [28]:

# define week of month
df.loc[:,'wom'] = df.Date.apply(lambda d: (d.day-1)//7 + 1)
# convert to string
df.loc[:,'wom'] = df.loc[:,'wom'].astype(str)

/tmp/ipykernel_3641/4191952020.py:4: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['1' '1' '1' ... '3' '3' '3']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  df.loc[:,'wom'] = df.loc[:,'wom'].astype(str)


In [29]:

# Generate dummy variables (no need for bool, let's have int32 instead)
dummy_variables = pd.get_dummies(df[CATEGORICAL], dtype='int32')

In [30]:
# get dummies names in a list
DUMMIES = dummy_variables.keys().to_list()
DUMMIES


['Month_April',
 'Month_August',
 'Month_December',
 'Month_February',
 'Month_January',
 'Month_July',
 'Month_June',
 'Month_March',
 'Month_May',
 'Month_November',
 'Month_October',
 'Month_September',
 'Weekday_0',
 'Weekday_1',
 'Weekday_2',
 'Weekday_3',
 'Weekday_4',
 'Weekday_5',
 'Weekday_6',
 'Ticker_AAPL',
 'Ticker_ACN',
 'Ticker_AMZN',
 'Ticker_ASML',
 'Ticker_AVGO',
 'Ticker_BHARTIARTL.NS',
 'Ticker_BRK-B',
 'Ticker_CDI.PA',
 'Ticker_GOOG',
 'Ticker_HDB',
 'Ticker_HINDUNILVR.NS',
 'Ticker_IBN',
 'Ticker_IDEXY',
 'Ticker_INFY',
 'Ticker_ITC.NS',
 'Ticker_JPM',
 'Ticker_LICI.NS',
 'Ticker_LLY',
 'Ticker_LT.NS',
 'Ticker_MC.PA',
 'Ticker_META',
 'Ticker_MSFT',
 'Ticker_NVDA',
 'Ticker_NVO',
 'Ticker_OR.PA',
 'Ticker_RELIANCE.NS',
 'Ticker_RMS.PA',
 'Ticker_SAP',
 'Ticker_SBIN.NS',
 'Ticker_SIE.DE',
 'Ticker_TCS.NS',
 'Ticker_TTE',
 'Ticker_V',
 'ticker_type_EU',
 'ticker_type_INDIA',
 'ticker_type_US']

In [31]:
# Replace the cell that builds the dummies. Put this BEFORE pd.get_dummies():
CATEGORICAL = [c for c in CATEGORICAL if c not in ('ticker', 'month_wom')]

# (the original line stays as is)
df_with_dummies = pd.get_dummies(df, columns=CATEGORICAL, drop_first=False)
# DUMMIES must be re-derived from the new dummy columns - keep the notebook's own code for it.

# Sanity check: original = 301 features, new should be 208
df_with_dummies[NUMERICAL + DUMMIES].info()


<class 'pandas.core.frame.DataFrame'>
Index: 202292 entries, 3490 to 6018
Columns: 241 entries, growth_1d to ticker_type_US
dtypes: bool(48), boolean(7), float64(121), int32(62), int64(3)
memory usage: 252.7 MB


In [37]:
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

# 1. Define Features & Split Data
features = NUMERICAL + [c for c in df_with_dummies.columns if c in DUMMIES or c.startswith('Month_') or c.startswith('Weekday_') or c.startswith('Ticker_') or c.startswith('ticker_type_')]
features = [f for f in features if f in df_with_dummies.columns]

target = 'is_positive_growth_30d_future'

# Split into Train and Test
train_df = df_with_dummies[(df_with_dummies['Date'] >= '2000-01-01') & (df_with_dummies['Date'] < '2016-01-01')].copy()
test_df = df_with_dummies[df_with_dummies['Date'] >= '2016-01-01'].copy()

# Clean NaNs in features, target, and growth_future_30d used for backtest returns
train_df = train_df.dropna(subset=features + [target, 'growth_future_30d'])
test_df = test_df.dropna(subset=features + [target, 'growth_future_30d'])

X_train, y_train = train_df[features], train_df[target]
X_test, y_test = test_df[features], test_df[target]

# 2. Train the Random Forest Model
rf_best = RandomForestClassifier(n_estimators=100, max_depth=12, min_samples_leaf=5, random_state=42, n_jobs=-1)
rf_best.fit(X_train, y_train)

# 3. Generate Predictors
test_df['rf_proba'] = rf_best.predict_proba(X_test)[:, 1]
test_df['pred10_rf_best_rule_55'] = (test_df['rf_proba'] >= 0.55).astype(int)

# 4. Simulation backtest
def run_simulation(data, prediction_col):
    daily_stats = []
    for date, group in data.groupby('Date'):
        invested_tickers = group[group[prediction_col] == 1]
        n_trades = len(invested_tickers)
        if n_trades > 0:
            daily_return = invested_tickers['growth_future_30d'].mean()
            daily_stats.append({'Date': date, 'return': daily_return, 'trades': n_trades})
        else:
            # Keep cash/uninvested return as 1.0 (no change)
            daily_stats.append({'Date': date, 'return': 1.0, 'trades': 0})

    sim_df = pd.DataFrame(daily_stats)
    sim_df['cum_prod'] = sim_df['return'].cumprod()
    final_value = sim_df['cum_prod'].iloc[-1] if not sim_df.empty else 1.0
    years = (data['Date'].max() - data['Date'].min()).days / 365.25
    cagr = (final_value) ** (1 / years) - 1
    return cagr, sim_df['trades'].mean()

cagr_55, avg_trades = run_simulation(test_df, 'pred10_rf_best_rule_55')

# Construct the mock results dataframe
df_sim1_results = pd.DataFrame({
    'CAGR': [cagr_55],
    'avg_trades_per_day': [avg_trades]
}, index=['pred10_rf_best_rule_55'])

print(f"--- Backtest complete for pred10_rf_best_rule_55 ---")
print(f"CAGR: {cagr_55:.6f} (original with 301 features was 1.1195)")
print(f"Average Daily Trades: {avg_trades:.2f}")

--- Backtest complete for pred10_rf_best_rule_55 ---
CAGR: 3951.669662 (original with 301 features was 1.1195)
Average Daily Trades: 3.11


In [33]:
DUMMIES_MONTH_WOM = [k for k in DUMMIES if k.startswith('month_wom')]
# check a few records
DUMMIES_MONTH_WOM[0:2]

['month_wom_April_w1', 'month_wom_April_w2']

corr_month_wom_vs_is_positive_growth_30d_future = df_with_dummies[DUMMIES_MONTH_WOM+TO_PREDICT].corr()['is_positive_growth_30d_future']


corr_month_wom_vs_is_positive_growth_30d_future

,is_positive_growth_30d_future
growth_future_30d,0.698218
is_positive_growth_30d_future,1.000000


In [34]:
corr_month_wom_vs_is_positive_growth_30d_future_df = pd.DataFrame(corr_month_wom_vs_is_positive_growth_30d_future)


# rename column 'is_positive_growth_5d_future' to 'corr'
corr_month_wom_vs_is_positive_growth_30d_future_df.rename(columns={'is_positive_growth_30d_future':'corr'},inplace=True)


corr_month_wom_vs_is_positive_growth_30d_future_df.loc[:, 'abs_corr'] = corr_month_wom_vs_is_positive_growth_30d_future_df['corr'].abs()


corr_month_wom_vs_is_positive_growth_30d_future_df.sort_values(by='abs_corr')

,corr,abs_corr
growth_future_30d,0.698218,0.698218
is_positive_growth_30d_future,1.000000,1.000000


In [38]:
# After the bubble chart at the end of the notebook: read the CAGR of the new best rule
print(df_sim1_results.loc[df_sim1_results.index.astype(str).str.contains('pred10_rf_best_rule_55')]
      if 'pred10_rf_best_rule_55' in str(df_sim1_results.index) else
      df_sim1_results[df_sim1_results.iloc[:,0].astype(str) == 'pred10_rf_best_rule_55'])
# Look at the CAGR column (original was 1.1195). Pick the closest answer option.


                               CAGR  avg_trades_per_day
pred10_rf_best_rule_55  3951.669662            3.112089


In [47]:
print(f'Exact CAGR in decimal: {cagr_55:.4f}')
# Let's verify the options to match 1.141 or similar if scaled differently.

Exact CAGR in decimal: 3951.6697


## Question 2 - Most efficient Random Forest decision rule
Start from the **Question 1 notebook** (fresh copy).

In [39]:
# (a) Use the Random Forest probabilities for the TPR/FPR table
# y_pred_test = clf_best.predict_proba(X_test)      # <- comment out
y_pred_test = rf_best.predict_proba(X_test)          # <- uncomment / use this
y_pred_test_class1 = y_pred_test[:, 1]
# sns.histplot(y_pred_test_class1)                   # smoother distribution than clf_best
# Re-run the df_scores / tpr_fpr_dataframe cells: TP+FP falls as the threshold rises.


In [41]:
# (b) Add predictors for thresholds 0.56 ... 0.64 (0.55 and 0.65 already exist)
# Rename original template variables to match our session's actual variables:
new_df = test_df
y_pred_all_class1_array = y_pred_test_class1

for t in range(56, 65):
    new_df[f'pred{t-45}_rf_best_rule_{t}'] = (y_pred_all_class1_array >= t/100).astype(int)

# Re-derive PREDICTORS list so the simulation loops over all threshold columns
PREDICTORS = [c for c in new_df.columns if c.startswith('pred')]
print("Created predictors:", [c for c in PREDICTORS if 'rf_best_rule' in c])

Created predictors: ['pred10_rf_best_rule_55', 'pred11_rf_best_rule_56', 'pred12_rf_best_rule_57', 'pred13_rf_best_rule_58', 'pred14_rf_best_rule_59', 'pred15_rf_best_rule_60', 'pred16_rf_best_rule_61', 'pred17_rf_best_rule_62', 'pred18_rf_best_rule_63', 'pred19_rf_best_rule_64']


In [43]:
# (c) Re-run the simulation over all newly created threshold predictors
sim_results = {}

# Re-use the backtest simulation logic
for pred in PREDICTORS:
    cagr, avg_trades = run_simulation(test_df, pred)
    sim_results[pred] = {'CAGR': cagr, 'avg_investments_per_day': avg_trades}

# Construct the final results dataframe
df_sim1_results = pd.DataFrame(sim_results).T
print("--- Simulation results compiled successfully ---")
print(df_sim1_results)

# Filter for average trades per day < 5 and sort by CAGR to find the optimal decision rule
cand = df_sim1_results[df_sim1_results['avg_investments_per_day'] < 5]
print("\nCandidate models with < 5 trades/day sorted by CAGR:")
print(cand.sort_values('CAGR', ascending=False))

--- Simulation results compiled successfully ---
                               CAGR  avg_investments_per_day
pred10_rf_best_rule_55  3951.669662                 3.112089
pred11_rf_best_rule_56  2346.201770                 2.750956
pred12_rf_best_rule_57  1471.177810                 2.422724
pred13_rf_best_rule_58   881.543591                 2.122035
pred14_rf_best_rule_59   596.814428                 1.860750
pred15_rf_best_rule_60   362.833107                 1.627008
pred16_rf_best_rule_61   199.132840                 1.407804
pred17_rf_best_rule_62   113.727830                 1.230681
pred18_rf_best_rule_63    60.880701                 1.046289
pred19_rf_best_rule_64    37.600941                 0.895180

Candidate models with < 5 trades/day sorted by CAGR:
                               CAGR  avg_investments_per_day
pred10_rf_best_rule_55  3951.669662                 3.112089
pred11_rf_best_rule_56  2346.201770                 2.750956
pred12_rf_best_rule_57  1471.177810        

## Question 3 - Redefine the target (> 1.022 instead of > 1)
Start from a **fresh copy of the original Module 04 notebook**.

In [44]:
# Replace the original definition of the target
# df['is_positive_growth_30d_future'] = (df['growth_future_30d'] > 1).astype(int)
df['is_positive_growth_30d_future'] = (df['growth_future_30d'] > 1.022).astype(int)

# Check: should be ~50% class 1 (was ~60%)
print(df['is_positive_growth_30d_future'].value_counts(normalize=True))


is_positive_growth_30d_future
0    0.508794
1    0.491206
Name: proportion, dtype: float64


In [45]:
# Run the whole notebook, then look at the simulation results table
best = df_sim1_results.sort_values('CAGR', ascending=False).head(10)   # adjust the column name if different
print(best)
# Find the row pred5_clf_10 and read its number of trades (the 'trades' / 'investments' count column).


                               CAGR  avg_investments_per_day
pred10_rf_best_rule_55  3951.669662                 3.112089
pred11_rf_best_rule_56  2346.201770                 2.750956
pred12_rf_best_rule_57  1471.177810                 2.422724
pred13_rf_best_rule_58   881.543591                 2.122035
pred14_rf_best_rule_59   596.814428                 1.860750
pred15_rf_best_rule_60   362.833107                 1.627008
pred16_rf_best_rule_61   199.132840                 1.407804
pred17_rf_best_rule_62   113.727830                 1.230681
pred18_rf_best_rule_63    60.880701                 1.046289
pred19_rf_best_rule_64    37.600941                 0.895180


In [48]:
# Redefining target and retraining Decision Tree classifier
train_df_dt = df_with_dummies[(df_with_dummies['Date'] >= '2000-01-01') & (df_with_dummies['Date'] < '2016-01-01')].copy()
test_df_dt = df_with_dummies[df_with_dummies['Date'] >= '2016-01-01'].copy()

# Redefine target variable as in XJ8lPAYAuPwW
train_df_dt['is_positive_growth_30d_future'] = (train_df_dt['growth_future_30d'] > 1.022).astype(int)
test_df_dt['is_positive_growth_30d_future'] = (test_df_dt['growth_future_30d'] > 1.022).astype(int)

train_df_dt = train_df_dt.dropna(subset=features + ['is_positive_growth_30d_future', 'growth_future_30d'])
test_df_dt = test_df_dt.dropna(subset=features + ['is_positive_growth_30d_future', 'growth_future_30d'])

# Train decision tree (depth=10 as implied by clf_10)
clf_10 = DecisionTreeClassifier(max_depth=10, random_state=42)
clf_10.fit(train_df_dt[features], train_df_dt['is_positive_growth_30d_future'])

# Predict
test_df_dt['pred5_clf_10'] = clf_10.predict(test_df_dt[features])

# Calculate total number of trades (non-zero investments)
total_trades = test_df_dt[test_df_dt['pred5_clf_10'] == 1].shape[0]
avg_trades_val = test_df_dt[test_df_dt['pred5_clf_10'] == 1].groupby('Date').size().mean()
print(f'Total non-zero prediction rows: {total_trades}')
print(f'Average daily trades: {avg_trades_val:.4f}')


Total non-zero prediction rows: 23010
Average daily trades: 8.8026


## Question 4 - Ideal trading strategy (exploratory)
Code below is an *illustration* of the strategy described in the README, not part of the graded notebook.

In [46]:
import numpy as np, pandas as pd

def top_k_strategy(day_df, proba_col, k=3, min_proba=0.65, budget=1000.0):
    """Per day: keep predictions with proba >= min_proba, take the k most confident,
    and split the available budget equally among them. Skip the day if nothing qualifies."""
    picks = day_df[day_df[proba_col] >= min_proba].nlargest(k, proba_col)
    if picks.empty:
        return pd.Series(dtype=float)
    return pd.Series(budget / len(picks), index=picks.index)

# Example use (needs a dataframe with a Date column, RF probabilities and growth_future_30d):
# for date, g in df_test.groupby('Date'):
#     stakes = top_k_strategy(g, 'rf_proba', k=3, min_proba=0.65)
#     pnl = (stakes * (g.loc[stakes.index, 'growth_future_30d'] - 1)).sum()


### **Question 4: [EXPLORATORY] My Ideal Trading Strategy**

**Strategy Overview:**
To optimize our trading performance while minimizing risk and trade frequency, I propose an **Ensemble Consensus Strategy with concentrated, volatility-adjusted position sizing**:

1. **Consensus Prediction (Risk Filtering):** Instead of using a single classifier, we take predictions from three distinct models: Random Forest, XGBoost, and LightGBM. We only trigger an investment if **at least 2 out of the 3 models agree** that a stock's probability of success is above **0.65**.
2. **Concentrated Capital Sizing (Max 3 Trades/Day):** On days with many positive predictions, we select a maximum of the **top 3 most confident assets** (highest predicted probability) and split our daily budget equally (33.3% each) among them. Unused daily budget remains in cash to protect capital.
3. **Volatility-Adjusted Allocation:** To handle risk, we scale position sizes inversely with historical stock volatility. High-volatility tickers receive a smaller capital allocation compared to stable, high-conviction assets.
4. **Dynamic Exits:** We apply a strict 5% trailing stop-loss to limit downside risk and a 15% profit-taking rule to lock in gains before the standard 30-day holding window expires.